In [ ]:
import os
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

from ss_gan.forward import RickerWavelet, forward_seismic_from_impedance
from ss_gan.models import UNet1D

# Paths (edit if needed)
ROOT = Path('..').resolve()
DATASET = ROOT / 'data' / 'marmousi2_2721_like_l101.npz'
# Example checkpoint path (edit to a specific run if desired)
CKPT = ROOT / 'runs' / 'ablations_fixscale_050' / 'A_norm_mse' / 'checkpoints' / 'best.pt'

print('ROOT:', ROOT)
print('DATASET exists:', DATASET.exists(), DATASET)
print('CKPT exists:', CKPT.exists(), CKPT)

## 1) 读取 NPZ 并检查 shape / split / stats

In [ ]:
z = np.load(DATASET, allow_pickle=True)
keys = list(z.keys())
print('keys:', keys)

def show_arr(name: str):
    a = z[name]
    print(f'{name}: shape={a.shape}, dtype={a.dtype}, min={np.min(a):.4g}, max={np.max(a):.4g}, mean={np.mean(a):.4g}, std={np.std(a):.4g}')

for name in ['x_labeled','y_labeled','x_unlabeled','x_val','y_val','x_test','y_test']:
    if name in z:
        show_arr(name)

stats = None
if 'stats' in z:
    stats = z['stats'].item()
    print('stats:', stats)

# sanity: shapes
assert z['x_labeled'].ndim == 2 and z['y_labeled'].ndim == 2
assert z['x_labeled'].shape[1] == z['y_labeled'].shape[1]
print('OK: labeled x/y length aligned')

## 2) 正演模块（阻抗→地震）快速验证
这里用 1 条阻抗曲线跑正演，检查输出长度/数值范围是否合理。

In [ ]:
# take a single trace
y0 = z['y_labeled'][0:1]  # [1,T]
T = y0.shape[1]

# torch tensors: [B,1,T]
y0_t = torch.from_numpy(y0).float().unsqueeze(1)

wavelet = RickerWavelet(freq_hz=30.0, dt=0.001, duration_s=0.128, device='cpu').tensor()
s0 = forward_seismic_from_impedance(y0_t, wavelet).squeeze(0).squeeze(0).numpy()

print('y0:', y0.shape, 's0:', s0.shape)
assert s0.shape[0] == T

plt.figure(figsize=(10,3))
plt.plot(y0[0], label='impedance (one trace)')
plt.title('Impedance trace (raw units)')
plt.legend(); plt.tight_layout()
plt.show()

plt.figure(figsize=(10,3))
plt.plot(s0, label='forward seismic')
plt.title('Forward seismic from impedance (one trace)')
plt.legend(); plt.tight_layout()
plt.show()

## 3) UNet1D forward（shape 检查 + 简单数值检查）

In [ ]:
# Build a model with default paper-like params
G = UNet1D(in_ch=1, out_ch=1, base_ch=16, k_large=299, k_small=3).eval()

x0 = z['x_labeled'][0:2]  # [2,T]
x0_t = torch.from_numpy(x0).float().unsqueeze(1)  # [2,1,T]

with torch.no_grad():
    y_pred = G(x0_t)

print('x0_t:', tuple(x0_t.shape), 'y_pred:', tuple(y_pred.shape))
assert y_pred.shape == x0_t.shape
print('OK: UNet1D output shape matches input')

## 4)（可选）加载 checkpoint，做少量推理并画 4 道对比
如果 `CKPT` 不存在，把上面路径改成你某个 run 的 `checkpoints/best.pt`。

In [ ]:
if not CKPT.exists():
    raise FileNotFoundError(f'Checkpoint not found: {CKPT}')

ckpt = torch.load(CKPT, map_location='cpu')
cfg = ckpt.get('cfg', {})
stats_ckpt = ckpt.get('stats', {}) or {}
print('ckpt cfg keys:', list(cfg.keys())[:20])
print('ckpt stats:', {k: stats_ckpt.get(k) for k in ['x_mean','x_std','y_mean','y_std']})

k_large = int(cfg.get('k_large', 299))
k_small = int(cfg.get('k_small', 3))
base_ch_g = int(cfg.get('base_ch_g', 16))
normalize = bool(cfg.get('normalize', True))

G = UNet1D(1, 1, base_ch_g, k_large, k_small)
G.load_state_dict(ckpt['G'])
G.eval()

# Prepare a small batch for plotting
trace_ids_1based = [299, 2299, 599, 1699]
idxs = [t-1 for t in trace_ids_1based]
x = z['x_test'][idxs]  # [4,T]
y = z['y_test'][idxs]  # [4,T]

# If model was trained with normalize=true, mimic infer.py behavior: apply z-score using checkpoint stats
if normalize and ('x_mean' in stats_ckpt) and ('x_std' in stats_ckpt):
    x_in = (x - float(stats_ckpt['x_mean'])) / (float(stats_ckpt['x_std']) + 1e-12)
else:
    x_in = x

with torch.no_grad():
    pred = G(torch.from_numpy(x_in).float().unsqueeze(1)).squeeze(1).numpy()  # [4,T]

# Plot in physical units if possible
if normalize and ('y_mean' in stats_ckpt) and ('y_std' in stats_ckpt):
    pred_plot = pred * float(stats_ckpt['y_std']) + float(stats_ckpt['y_mean'])
    y_plot = y
else:
    pred_plot = pred
    y_plot = y

t = np.arange(pred_plot.shape[1])
fig, axes = plt.subplots(2, 2, figsize=(12.5, 7.0))
axes = axes.reshape(-1)
for ax, tr, p1, y1 in zip(axes, trace_ids_1based, pred_plot, y_plot):
    ax.plot(t, p1, color='red', lw=1.5, label='pred')
    ax.plot(t, y1, color='blue', lw=1.5, label='true')
    ax.set_title(f'Trace {tr}')
    ax.legend(loc='upper left')
plt.tight_layout()
plt.show()